# 🐟 Fish Speech — Google Colab Runner

**Model:** Fish Audio S2 Pro (5B Parameters)

**What this notebook gives you:**
- ✅ Section 1 — Install Fish Speech code and clone S2-Pro weights
- ✅ Section 2 — FastAPI REST **API endpoint** with public ngrok URL

> ⚠️ **Enable GPU before running!**  
> `Runtime → Change runtime type → T4 GPU`

---
## 🔧 Section 1 — Install Dependencies and Model
This step clones the code and directly clones the S2 Pro weights from HuggingFace.

In [ ]:
# Ensure we start in the right directory
%cd /content

# Install system dependencies required for pyaudio and tokenizers (rust)
!apt-get update -qq
!apt-get install -y portaudio19-dev build-essential rustc cargo git git-lfs

# Clean up previous installations if they exist, then clone
!rm -rf fish-speech
!git clone https://github.com/fishaudio/fish-speech.git
%cd /content/fish-speech

# --- APPLY MEMORY OOM FIX ---
# We inject a patch into Fish Speech's code to force it to use bfloat16 when initializing.
# PyTorch defaults to float32 (20GB), which crashes Colab. bfloat16 shrinks it to 10GB.
# This completely prevents the memory crash without causing the Meta tensor bugs!
import os
llama_path = "fish_speech/models/text2semantic/llama.py"
with open(llama_path, "r") as f:
    code = f.read()

code = code.replace(
    "model = model_cls(config)",
    "torch.set_default_dtype(torch.bfloat16)\n        model = model_cls(config)\n        torch.set_default_dtype(torch.float32)"
)

# Fix CUDA OOM by restricting max sequence length to 4096 (saves 4.5 GB of GPU VRAM allocated to KV Cache!)
code = code.replace(
    "config = BaseModelArgs.from_pretrained(str(path))",
    "config = BaseModelArgs.from_pretrained(str(path))\n        config.max_seq_len = 4096"
)
with open(llama_path, "w") as f:
    f.write(code)
print("✅ Out-of-Memory (OOM) fix successfully applied to Fish Speech source code!")
# ----------------------------

# Upgrade pip to ensure it pulls binary wheels instead of building from source
!python -m pip install -q --upgrade pip wheel
!pip install -q tokenizers transformers

# Install dependencies safely and FORCE torchvision downgrade to match Fish Speech's PyTorch version
!pip install -q -e . torchvision
!pip install -q fastapi uvicorn pyngrok nest_asyncio pyrootutils

# Clone the 5B S2 Pro model directly from HuggingFace
!git lfs install
!git clone https://huggingface.co/fishaudio/s2-pro checkpoints/s2-pro

print("✅ Dependencies and Model installed!")

---
## 🚀 Section 2 — Launch API Server with ngrok

> 🔑 **You need a free ngrok account.**  
> Add your token to Colab Secrets (the 🔑 icon on the left) as `NGROK_TOKEN`.
> Alternatively, paste it directly into the code below!

In [ ]:
# REPLACE "YOUR_TOKEN_HERE" WITH YOUR ACTUAL NGROK TOKEN IF SECRETS ARE NOT WORKING
MANUAL_TOKEN = "3HBO4T22yOJRKgg5yO9k00Ajicb_5SMadgykMTqqYKiTP38yC"

from google.colab import userdata
try:
    if MANUAL_TOKEN:
        ngrok_token = MANUAL_TOKEN
    else:
        ngrok_token = userdata.get("NGROK_TOKEN")
        
    if not ngrok_token:
        raise ValueError("Token is empty!")
        
    !ngrok authtoken {ngrok_token}
    print("✅ ngrok authenticated")
except Exception as e:
    print("\n❌ FATAL ERROR: Could not authenticate with ngrok!")
    print("You have two options to fix this:")
    print("1. Paste your token between the quotes in MANUAL_TOKEN = \"\" at the top of this cell.")
    print("2. OR Add your ngrok token to Colab Secrets (the 🔑 icon on the left) as NGROK_TOKEN and turn the toggle switch ON.")
    raise Exception("STOPPING: You must provide a valid ngrok token before continuing!")

In [ ]:
import nest_asyncio
import subprocess
import time
import os
from pyngrok import ngrok

nest_asyncio.apply()

# Ensure any zombie ngrok tunnels from previous interrupted runs are killed
ngrok.kill()
os.system("killall -9 ngrok 2>/dev/null")
time.sleep(1)

# Start ngrok tunnel
public_tunnel = ngrok.connect(8080)
public_url = public_tunnel.public_url

print("\n" + "="*60)
print("🚀 FISH SPEECH API is LIVE")
print("="*60)
print(f"  Public URL : {public_url}")
print(f"  API Route  : POST {public_url}/v1/tts")
print("="*60)
print("⏳ Starting backend server using the cloned 5B S2 Pro model...")

# Force working directory to the repo so 'tools.api_server' is found
%cd /content/fish-speech

# Launch the official Fish Audio API server using the downloaded S2 Pro model (using --half to save memory)
!python -m tools.api_server --listen 0.0.0.0:8080 --half
